In [1]:
from pathlib import Path
import json
import copy
import random
import time

import numpy as np
import pandas as pd

import torch
import torch.nn as nn
from torch.utils.data import Dataset, DataLoader

from sklearn.metrics import roc_auc_score, f1_score

print("PyTorch:", torch.__version__)

PyTorch: 2.14.0+cu130


In [2]:
DEVICE = torch.device(
    "cuda" if torch.cuda.is_available() else "cpu"
)

print("Device:", DEVICE)

if torch.cuda.is_available():
    print("GPU:", torch.cuda.get_device_name(0))
    print("CUDA:", torch.version.cuda)

Device: cuda
GPU: NVIDIA GeForce RTX 4060 Laptop GPU
CUDA: 13.0


In [3]:
SEED = 42

random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)

if torch.cuda.is_available():
    torch.cuda.manual_seed_all(SEED)

print("Seed:", SEED)

Seed: 42


In [4]:
PROJECT_ROOT = Path("../../").resolve()

PROCESSED_DIR = (
    PROJECT_ROOT
    / "data"
    / "processed"
    / "ecg"
    / "ptbxl"
)

MANIFEST_PATH = (
    PROCESSED_DIR
    / "ptbxl_manifest.csv"
)

CHECKPOINT_DIR = (
    PROJECT_ROOT
    / "checkpoints"
    / "ecg"
)

RESULTS_DIR = (
    PROJECT_ROOT
    / "artifacts"
    / "metrics"
    / "ecg"
)

CHECKPOINT_DIR.mkdir(
    parents=True,
    exist_ok=True
)

RESULTS_DIR.mkdir(
    parents=True,
    exist_ok=True
)

print("Manifest:", MANIFEST_PATH)
print("Exists:", MANIFEST_PATH.exists())

Manifest: D:\Programming\VS Code\Projects\CardioFusion-XAI\ml-service\data\processed\ecg\ptbxl\ptbxl_manifest.csv
Exists: True


In [5]:
ptbxl = pd.read_csv(
    MANIFEST_PATH
)

print("Manifest shape:", ptbxl.shape)

display(
    ptbxl.head()
)

Manifest shape: (21837, 20)


,record_id,patient_id,record_name,processed_path,split,original_fs,NORM,MI,STTC,CD,HYP,RHYTHM_SR,RHYTHM_AFIB,RHYTHM_AFLT,RHYTHM_STACH,RHYTHM_SBRAD,RHYTHM_SARRH,RHYTHM_PSVT,RHYTHM_BIGU,RHYTHM_PACE
0,1,15709.0,records100/00000/00001_lr,data\processed\ecg\ptbxl\waveforms\00001_lr.npy,test,100,1.0,0.0,0.0,0.0,0.0,1.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
1,2,13243.0,records100/00000/00002_lr,data\processed\ecg\ptbxl\waveforms\00002_lr.npy,train,100,1.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,1.0,0.0,0.0,0.0,0.0
2,3,20372.0,records100/00000/00003_lr,data\processed\ecg\ptbxl\waveforms\00003_lr.npy,train,100,1.0,0.0,0.0,0.0,0.0,1.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
3,4,17014.0,records100/00000/00004_lr,data\processed\ecg\ptbxl\waveforms\00004_lr.npy,test,100,1.0,0.0,0.0,0.0,0.0,1.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
4,5,17448.0,records100/00000/00005_lr,data\processed\ecg\ptbxl\waveforms\00005_lr.npy,train,100,1.0,0.0,0.0,0.0,0.0,1.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0


In [6]:
DIAGNOSTIC_LABELS = [
    "NORM",
    "MI",
    "STTC",
    "CD",
    "HYP",
]

diagnostic_targets = [
    col
    for col in DIAGNOSTIC_LABELS
    if col in ptbxl.columns
]

rhythm_targets = [
    col
    for col in ptbxl.columns
    if col.startswith("RHYTHM_")
]

print("Diagnostic targets:")
print(diagnostic_targets)

print("\nRhythm targets:")
print(rhythm_targets)

if not diagnostic_targets:
    raise ValueError(
        "No diagnostic targets found in the manifest."
    )

Diagnostic targets:
['NORM', 'MI', 'STTC', 'CD', 'HYP']

Rhythm targets:
['RHYTHM_SR', 'RHYTHM_AFIB', 'RHYTHM_AFLT', 'RHYTHM_STACH', 'RHYTHM_SBRAD', 'RHYTHM_SARRH', 'RHYTHM_PSVT', 'RHYTHM_BIGU', 'RHYTHM_PACE']


In [7]:
required_columns = [
    "processed_path",
    "split",
]

missing = [
    col
    for col in required_columns
    if col not in ptbxl.columns
]

if missing:
    raise ValueError(
        f"Missing columns: {missing}"
    )

ptbxl = ptbxl[
    ptbxl["split"].isin(
        ["train", "val", "test"]
    )
].copy()

ptbxl["processed_path"] = (
    ptbxl["processed_path"]
    .astype(str)
)

print("Usable records:", len(ptbxl))

print("\nSplit counts:")
print(
    ptbxl["split"]
    .value_counts()
)

Usable records: 21837

Split counts:
split
train    15298
test      3273
val       3266
Name: count, dtype: int64


In [8]:
NUM_LEADS = 12
SIGNAL_LENGTH = 1000

BATCH_SIZE = 32

NUM_WORKERS = 0

EPOCHS = 30

LEARNING_RATE = 1e-4
WEIGHT_DECAY = 1e-4

print("Leads:", NUM_LEADS)
print("Signal length:", SIGNAL_LENGTH)
print("Batch size:", BATCH_SIZE)

Leads: 12
Signal length: 1000
Batch size: 32


In [9]:
class PTBXLDataset(Dataset):

    def __init__(
        self,
        dataframe,
        project_root,
        diagnostic_targets,
        rhythm_targets,
    ):
        self.df = dataframe.reset_index(
            drop=True
        )

        self.project_root = Path(
            project_root
        )

        self.diagnostic_targets = (
            diagnostic_targets
        )

        self.rhythm_targets = (
            rhythm_targets
        )

        self.target_columns = (
            diagnostic_targets
            + rhythm_targets
        )

    def __len__(self):
        return len(self.df)

    def __getitem__(self, index):

        row = self.df.iloc[index]

        path = (
            self.project_root
            / row["processed_path"]
        )

        signal = np.load(
            path
        ).astype(np.float32)

        if signal.shape != (
            SIGNAL_LENGTH,
            NUM_LEADS
        ):
            raise ValueError(
                f"Unexpected ECG shape "
                f"{signal.shape} for {path}"
            )

        # (1000, 12) -> (12, 1000)
        signal = signal.T

        diagnostic = np.array(
            [
                row[col]
                for col in self.diagnostic_targets
            ],
            dtype=np.float32
        )

        if self.rhythm_targets:

            rhythm = np.array(
                [
                    row[col]
                    for col in self.rhythm_targets
                ],
                dtype=np.float32
            )

        else:

            rhythm = np.empty(
                0,
                dtype=np.float32
            )

        return (
            torch.tensor(
                signal,
                dtype=torch.float32
            ),
            torch.tensor(
                diagnostic,
                dtype=torch.float32
            ),
            torch.tensor(
                rhythm,
                dtype=torch.float32
            ),
        )

In [10]:
train_df = ptbxl[
    ptbxl["split"] == "train"
].copy()

val_df = ptbxl[
    ptbxl["split"] == "val"
].copy()

test_df = ptbxl[
    ptbxl["split"] == "test"
].copy()

print("Train:", len(train_df))
print("Validation:", len(val_df))
print("Test:", len(test_df))

Train: 15298
Validation: 3266
Test: 3273


In [11]:
train_dataset = PTBXLDataset(
    train_df,
    PROJECT_ROOT,
    diagnostic_targets,
    rhythm_targets,
)

val_dataset = PTBXLDataset(
    val_df,
    PROJECT_ROOT,
    diagnostic_targets,
    rhythm_targets,
)

test_dataset = PTBXLDataset(
    test_df,
    PROJECT_ROOT,
    diagnostic_targets,
    rhythm_targets,
)

print("Datasets created.")

Datasets created.


In [12]:
train_loader = DataLoader(
    train_dataset,
    batch_size=BATCH_SIZE,
    shuffle=True,
    num_workers=NUM_WORKERS,
    pin_memory=torch.cuda.is_available(),
)

val_loader = DataLoader(
    val_dataset,
    batch_size=BATCH_SIZE,
    shuffle=False,
    num_workers=NUM_WORKERS,
    pin_memory=torch.cuda.is_available(),
)

test_loader = DataLoader(
    test_dataset,
    batch_size=BATCH_SIZE,
    shuffle=False,
    num_workers=NUM_WORKERS,
    pin_memory=torch.cuda.is_available(),
)

print("DataLoaders created.")

DataLoaders created.


In [13]:
signals, diagnostic, rhythm = next(
    iter(train_loader)
)

print("ECG batch:", signals.shape)
print("Diagnostic targets:", diagnostic.shape)
print("Rhythm targets:", rhythm.shape)

ECG batch: torch.Size([32, 12, 1000])
Diagnostic targets: torch.Size([32, 5])
Rhythm targets: torch.Size([32, 9])


In [14]:
from src.models.ecg.xresnet1d import XResNet1D

ModuleNotFoundError: No module named 'src'

In [ ]:
NUM_DIAGNOSTIC = len(
    diagnostic_targets
)

NUM_RHYTHM = len(
    rhythm_targets
)

model = XResNet1D(
    input_channels=12,
    num_diagnostic_classes=NUM_DIAGNOSTIC,
    num_rhythm_classes=NUM_RHYTHM,
)

model = model.to(
    DEVICE
)

print(model)

In [15]:
def calculate_pos_weights(
    dataframe,
    columns,
):
    weights = []

    for col in columns:

        positive = (
            dataframe[col] == 1
        ).sum()

        negative = (
            dataframe[col] == 0
        ).sum()

        if positive == 0:
            weight = 1.0
        else:
            weight = (
                negative
                / positive
            )

        weights.append(weight)

    return torch.tensor(
        weights,
        dtype=torch.float32,
        device=DEVICE
    )


diagnostic_pos_weight = calculate_pos_weights(
    train_df,
    diagnostic_targets,
)

print(
    "Diagnostic pos_weight:",
    diagnostic_pos_weight
)

Diagnostic pos_weight: tensor([1.2967, 3.0037, 3.1492, 3.4176, 7.1286], device='cuda:0')


In [16]:
if rhythm_targets:

    rhythm_pos_weight = (
        calculate_pos_weights(
            train_df,
            rhythm_targets,
        )
    )

    print(
        "Rhythm pos_weight:",
        rhythm_pos_weight
    )

else:

    rhythm_pos_weight = None

    print(
        "No rhythm targets."
    )

Rhythm pos_weight: tensor([2.9743e-01, 1.3681e+01, 2.8764e+02, 2.5106e+01, 3.3689e+01, 2.7756e+01,
        1.2738e+03, 2.6276e+02, 7.5874e+01], device='cuda:0')


In [17]:
diagnostic_criterion = (
    nn.BCEWithLogitsLoss(
        pos_weight=diagnostic_pos_weight
    )
)

if rhythm_targets:

    rhythm_criterion = (
        nn.BCEWithLogitsLoss(
            pos_weight=rhythm_pos_weight
        )
    )

else:

    rhythm_criterion = None

In [18]:
optimizer = torch.optim.AdamW(
    model.parameters(),
    lr=LEARNING_RATE,
    weight_decay=WEIGHT_DECAY,
)

scheduler = torch.optim.lr_scheduler.ReduceLROnPlateau(
    optimizer,
    mode="min",
    factor=0.5,
    patience=2,
)

NameError: name 'model' is not defined

In [19]:
USE_AMP = torch.cuda.is_available()

scaler = torch.amp.GradScaler(
    "cuda",
    enabled=USE_AMP
)

print(
    "Mixed precision:",
    USE_AMP
)

Mixed precision: True


In [ ]:
def train_one_epoch(
    model,
    loader,
    optimizer,
    scaler,
):
    model.train()

    total_loss = 0.0
    total_samples = 0

    for signals, diagnostic, rhythm in loader:

        signals = signals.to(
            DEVICE,
            non_blocking=True
        )

        diagnostic = diagnostic.to(
            DEVICE,
            non_blocking=True
        )

        rhythm = rhythm.to(
            DEVICE,
            non_blocking=True
        )

        optimizer.zero_grad(
            set_to_none=True
        )

        with torch.autocast(
            device_type=DEVICE.type,
            dtype=torch.float16,
            enabled=USE_AMP,
        ):

            outputs = model(
                signals
            )

            diagnostic_logits = (
                outputs["diagnostic"]
            )

            diagnostic_loss = (
                diagnostic_criterion(
                    diagnostic_logits,
                    diagnostic
                )
            )

            if rhythm_targets:

                rhythm_logits = (
                    outputs["rhythm"]
                )

                rhythm_loss = (
                    rhythm_criterion(
                        rhythm_logits,
                        rhythm
                    )
                )

                loss = (
                    diagnostic_loss
                    + rhythm_loss
                )

            else:

                loss = diagnostic_loss

        scaler.scale(
            loss
        ).backward()

        scaler.unscale_(
            optimizer
        )

        torch.nn.utils.clip_grad_norm_(
            model.parameters(),
            max_norm=1.0
        )

        scaler.step(
            optimizer
        )

        scaler.update()

        batch_size = signals.size(0)

        total_loss += (
            loss.item()
            * batch_size
        )

        total_samples += batch_size

    return (
        total_loss
        / total_samples
    )

In [ ]:
@torch.no_grad()
def evaluate(
    model,
    loader,
):
    model.eval()

    total_loss = 0.0
    total_samples = 0

    diagnostic_probs = []
    diagnostic_true = []

    rhythm_probs = []
    rhythm_true = []

    for signals, diagnostic, rhythm in loader:

        signals = signals.to(
            DEVICE,
            non_blocking=True
        )

        diagnostic = diagnostic.to(
            DEVICE,
            non_blocking=True
        )

        rhythm = rhythm.to(
            DEVICE,
            non_blocking=True
        )

        outputs = model(
            signals
        )

        diagnostic_logits = (
            outputs["diagnostic"]
        )

        diagnostic_loss = (
            diagnostic_criterion(
                diagnostic_logits,
                diagnostic
            )
        )

        if rhythm_targets:

            rhythm_logits = (
                outputs["rhythm"]
            )

            rhythm_loss = (
                rhythm_criterion(
                    rhythm_logits,
                    rhythm
                )
            )

            loss = (
                diagnostic_loss
                + rhythm_loss
            )

        else:

            loss = diagnostic_loss

        batch_size = signals.size(0)

        total_loss += (
            loss.item()
            * batch_size
        )

        total_samples += batch_size

        diagnostic_probs.extend(
            torch.sigmoid(
                diagnostic_logits
            )
            .cpu()
            .numpy()
            .tolist()
        )

        diagnostic_true.extend(
            diagnostic.cpu()
            .numpy()
            .tolist()
        )

        if rhythm_targets:

            rhythm_probs.extend(
                torch.sigmoid(
                    rhythm_logits
                )
                .cpu()
                .numpy()
                .tolist()
            )

            rhythm_true.extend(
                rhythm.cpu()
                .numpy()
                .tolist()
            )

    diagnostic_probs = np.asarray(
        diagnostic_probs
    )

    diagnostic_true = np.asarray(
        diagnostic_true
    )

    diagnostic_auc_values = []

    for i in range(
        diagnostic_probs.shape[1]
    ):

        if len(
            np.unique(
                diagnostic_true[:, i]
            )
        ) >= 2:

            diagnostic_auc_values.append(
                roc_auc_score(
                    diagnostic_true[:, i],
                    diagnostic_probs[:, i]
                )
            )

    diagnostic_auc = (
        float(
            np.mean(
                diagnostic_auc_values
            )
        )
        if diagnostic_auc_values
        else float("nan")
    )

    diagnostic_pred = (
        diagnostic_probs >= 0.5
    ).astype(int)

    diagnostic_f1 = f1_score(
        diagnostic_true,
        diagnostic_pred,
        average="macro",
        zero_division=0,
    )

    results = {
        "loss": total_loss / total_samples,
        "diagnostic_auroc": diagnostic_auc,
        "diagnostic_f1": diagnostic_f1,
    }

    if rhythm_targets:

        rhythm_probs = np.asarray(
            rhythm_probs
        )

        rhythm_true = np.asarray(
            rhythm_true
        )

        rhythm_auc_values = []

        for i in range(
            rhythm_probs.shape[1]
        ):

            if len(
                np.unique(
                    rhythm_true[:, i]
                )
            ) >= 2:

                rhythm_auc_values.append(
                    roc_auc_score(
                        rhythm_true[:, i],
                        rhythm_probs[:, i]
                    )
                )

        results["rhythm_auroc"] = (
            float(
                np.mean(
                    rhythm_auc_values
                )
            )
            if rhythm_auc_values
            else float("nan")
        )

        rhythm_pred = (
            rhythm_probs >= 0.5
        ).astype(int)

        results["rhythm_f1"] = f1_score(
            rhythm_true,
            rhythm_pred,
            average="macro",
            zero_division=0,
        )

    return results

In [ ]:
EPOCHS = 30
PATIENCE = 5

best_val_loss = float("inf")
best_state = None

patience_counter = 0

history = []

training_start = time.time()

print("=" * 75)
print("Starting PTB-XL XResNet1D training")
print("=" * 75)
print("Device:", DEVICE)
print("Training samples:", len(train_dataset))
print("Validation samples:", len(val_dataset))
print("Test samples:", len(test_dataset))
print("Batch size:", BATCH_SIZE)
print("Epochs:", EPOCHS)
print()

for epoch in range(
    1,
    EPOCHS + 1
):

    epoch_start = time.time()

    print(
        f"[Epoch {epoch}/{EPOCHS}] "
        f"Training started..."
    )

    train_loss = train_one_epoch(
        model,
        train_loader,
        optimizer,
        scaler,
    )

    print(
        f"[Epoch {epoch}/{EPOCHS}] "
        f"Training finished. "
        f"Running validation..."
    )

    val_results = evaluate(
        model,
        val_loader,
    )

    val_loss = val_results["loss"]

    scheduler.step(
        val_loss
    )

    epoch_time = (
        time.time()
        - epoch_start
    )

    total_time = (
        time.time()
        - training_start
    )

    history_row = {
        "epoch": epoch,
        "train_loss": train_loss,
        "val_loss": val_loss,
        "diagnostic_auroc": val_results[
            "diagnostic_auroc"
        ],
        "diagnostic_f1": val_results[
            "diagnostic_f1"
        ],
        "learning_rate": optimizer.param_groups[0]["lr"],
        "epoch_time_seconds": epoch_time,
    }

    if "rhythm_auroc" in val_results:

        history_row[
            "rhythm_auroc"
        ] = val_results[
            "rhythm_auroc"
        ]

        history_row[
            "rhythm_f1"
        ] = val_results[
            "rhythm_f1"
        ]

    history.append(
        history_row
    )

    print(
        f"[Epoch {epoch}/{EPOCHS}] "
        f"Train Loss: {train_loss:.4f} | "
        f"Val Loss: {val_loss:.4f}"
    )

    print(
        f"Diagnostic AUROC: "
        f"{val_results['diagnostic_auroc']:.4f} | "
        f"Diagnostic F1: "
        f"{val_results['diagnostic_f1']:.4f}"
    )

    if "rhythm_auroc" in val_results:

        print(
            f"Rhythm AUROC: "
            f"{val_results['rhythm_auroc']:.4f} | "
            f"Rhythm F1: "
            f"{val_results['rhythm_f1']:.4f}"
        )

    print(
        f"Epoch time: "
        f"{epoch_time / 60:.2f} min | "
        f"Total: "
        f"{total_time / 60:.2f} min"
    )

    if val_loss < best_val_loss:

        best_val_loss = val_loss

        best_state = copy.deepcopy(
            model.state_dict()
        )

        patience_counter = 0

        print(
            "✓ New best model."
        )

    else:

        patience_counter += 1

        print(
            f"No improvement "
            f"({patience_counter}/{PATIENCE})"
        )

        if patience_counter >= PATIENCE:

            print(
                "Early stopping."
            )

            break

    print("-" * 75)

print()
print("=" * 75)
print("Training complete")
print("=" * 75)
print(
    f"Total time: "
    f"{(time.time() - training_start) / 60:.2f} min"
)
print(
    f"Best validation loss: "
    f"{best_val_loss:.4f}"
)

In [ ]:
if not history:
    raise RuntimeError(
        "Training history is empty. "
        "Run the training cell first."
    )

history_df = pd.DataFrame(
    history
)

display(history_df)

plt.figure(figsize=(8, 5))

plt.plot(
    history_df["epoch"],
    history_df["train_loss"],
    label="Train Loss"
)

plt.plot(
    history_df["epoch"],
    history_df["val_loss"],
    label="Validation Loss"
)

plt.xlabel("Epoch")
plt.ylabel("Loss")
plt.title("PTB-XL XResNet1D Training")

plt.legend()

plt.tight_layout()
plt.show()

In [20]:
if best_state is None:
    raise RuntimeError(
        "No best model state was saved."
    )

model.load_state_dict(
    best_state
)

print(
    f"Restored best model "
    f"(validation loss = "
    f"{best_val_loss:.4f})"
)

NameError: name 'best_state' is not defined

In [21]:
test_results = evaluate(
    model,
    test_loader,
)

print("=" * 60)
print("PTB-XL TEST RESULTS")
print("=" * 60)

print(
    f"Test Loss          : "
    f"{test_results['loss']:.4f}"
)

print(
    f"Diagnostic AUROC   : "
    f"{test_results['diagnostic_auroc']:.4f}"
)

print(
    f"Diagnostic F1      : "
    f"{test_results['diagnostic_f1']:.4f}"
)

if "rhythm_auroc" in test_results:

    print(
        f"Rhythm AUROC      : "
        f"{test_results['rhythm_auroc']:.4f}"
    )

    print(
        f"Rhythm F1         : "
        f"{test_results['rhythm_f1']:.4f}"
    )

NameError: name 'evaluate' is not defined

In [22]:
@torch.no_grad()
def per_label_diagnostic_auc(
    model,
    loader,
):
    model.eval()

    all_probs = []
    all_true = []

    for signals, diagnostic, _ in loader:

        signals = signals.to(
            DEVICE
        )

        outputs = model(
            signals
        )

        probs = torch.sigmoid(
            outputs["diagnostic"]
        )

        all_probs.append(
            probs.cpu().numpy()
        )

        all_true.append(
            diagnostic.numpy()
        )

    all_probs = np.concatenate(
        all_probs,
        axis=0
    )

    all_true = np.concatenate(
        all_true,
        axis=0
    )

    rows = []

    for i, label in enumerate(
        diagnostic_targets
    ):

        if len(
            np.unique(
                all_true[:, i]
            )
        ) < 2:

            auc = np.nan

        else:

            auc = roc_auc_score(
                all_true[:, i],
                all_probs[:, i]
            )

        rows.append({
            "label": label,
            "AUROC": auc,
        })

    return pd.DataFrame(
        rows
    )


diagnostic_auc_df = (
    per_label_diagnostic_auc(
        model,
        test_loader
    )
)

display(
    diagnostic_auc_df
)

NameError: name 'model' is not defined

In [23]:
checkpoint_path = (
    CHECKPOINT_DIR
    / "ptbxl_xresnet1d101.pt"
)

checkpoint = {
    "model_state_dict": model.state_dict(),

    "architecture": "XResNet1D-101",

    "input_channels": NUM_LEADS,

    "input_length": SIGNAL_LENGTH,

    "diagnostic_targets": diagnostic_targets,

    "rhythm_targets": rhythm_targets,

    "sampling_rate_hz": 100,

    "duration_seconds": 10,

    "best_val_loss": best_val_loss,
}

torch.save(
    checkpoint,
    checkpoint_path
)

print("Checkpoint saved:")
print(checkpoint_path)

NameError: name 'model' is not defined

In [24]:
history_df.to_csv(
    RESULTS_DIR
    / "training_history.csv",
    index=False
)

test_summary = pd.DataFrame([
    test_results
])

test_summary.to_json(
    RESULTS_DIR
    / "test_metrics.json",
    orient="records",
    indent=2
)

diagnostic_auc_df.to_csv(
    RESULTS_DIR
    / "diagnostic_auc.csv",
    index=False
)

print(
    "Results saved to:",
    RESULTS_DIR
)

NameError: name 'history_df' is not defined

In [25]:
print("=" * 70)
print("PTB-XL TRAINING COMPLETE")
print("=" * 70)

print(
    "Architecture : XResNet1D-101"
)

print(
    f"Input        : "
    f"{NUM_LEADS} leads × "
    f"{SIGNAL_LENGTH} samples"
)

print(
    "Duration     : 10 seconds"
)

print(
    "Sampling     : 100 Hz"
)

print(
    f"Diagnostic   : "
    f"{len(diagnostic_targets)} labels"
)

print(
    f"Rhythm       : "
    f"{len(rhythm_targets)} labels"
)

print(
    f"Test AUROC   : "
    f"{test_results['diagnostic_auroc']:.4f}"
)

print(
    f"Test F1      : "
    f"{test_results['diagnostic_f1']:.4f}"
)

print(
    "Checkpoint   :",
    checkpoint_path
)

PTB-XL TRAINING COMPLETE
Architecture : XResNet1D-101
Input        : 12 leads × 1000 samples
Duration     : 10 seconds
Sampling     : 100 Hz
Diagnostic   : 5 labels
Rhythm       : 9 labels


NameError: name 'test_results' is not defined